# 🛒 DỰ ÁN DỰ ĐOÁN KHẢ NĂNG KHÁCH HÀNG QUAY LẠI MUA SẮM (CUSTOMER REPURCHASE PREDICTION)

## 📌 1. TỔNG QUAN ĐỀ TÀI
- **Loại bài toán:** Phân loại Nhị phân (Binary Classification) — Học máy có giám sát (Supervised ML).
- **Ý nghĩa thực tiễn:** Chi phí giữ chân khách hàng cũ thấp hơn **5 đến 7 lần** chi phí tìm kiếm khách hàng mới. Việc dự đoán sớm nguy cơ rời bỏ (Churn) giúp doanh nghiệp E-Commerce chủ động đưa ra ưu đãi, chăm sóc cá nhân hóa và tối ưu hóa doanh thu.
- **Dữ liệu:** Bộ dữ liệu thực tế từ Kaggle (*Ecommerce Customer Churn Analysis and Prediction* với **5,630 bản ghi**).
- **Nhãn Target ($y$):** `will_return` ($1$: Khách quay lại mua sắm trong 30 ngày, $0$: Khách rời đi / Churn).
- **Thuộc tính đặc trưng ($X$):** 8 đặc trưng RFM+ cốt lõi (`age`, `gender`, `total_purchases`, `avg_order_value`, `days_since_last_purchase`, `membership_level`, `used_voucher`, `satisfaction_score`).

In [ ]:
# === 1. TẢI CÁC THƯ VIỆN CẦN THIẾT ===
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print('Các thư viện đã được nạp thành công!')

## 📌 2. THU THẬP & ÁNH XẠ DỮ LIỆU THỰC TẾ (EDA & DATA MAPPING)

In [ ]:
# === 2. ĐỌC DỮ LIỆU KAGGLE THỰC TẾ & XỬ LÝ KHUYẾT ===
def load_dataset():
    paths = [
        '/kaggle/input/ecommerce-customer-churn-analysis-and-prediction/E Commerce Dataset.xlsx',
        'data/raw/E Commerce Dataset.xlsx',
        'E Commerce Dataset.xlsx'
    ]
    file_path = None
    for p in paths:
        if os.path.exists(p):
            file_path = p
            break
            
    if not file_path:
        raise FileNotFoundError('Vui lòng đảm bảo file E Commerce Dataset.xlsx đã được nạp vào tệp!')
        
    print(f'Loading data from: {file_path}')
    df_raw = pd.read_excel(file_path, sheet_name='E Comm')
    
    df = df_raw.copy()
    # Median imputation cho các cột khuyết
    num_cols = ['Tenure', 'OrderCount', 'DaySinceLastOrder', 'CouponUsed', 'OrderAmountHikeFromlastYear', 'CashbackAmount', 'SatisfactionScore']
    for c in num_cols:
        if c in df.columns:
            df[c] = df[c].fillna(df[c].median())
            
    # Ánh xạ nhãn target will_return (1 - Churn)
    df['will_return'] = (df['Churn'] == 0).astype(int)
    
    # Ánh xạ 8 thuộc tính đề tài
    df['gender'] = df['Gender'].replace({'Male': 'Nam', 'Female': 'Nữ'}).fillna('Nữ')
    df['age'] = np.clip((22 + df['Tenure'] * 0.8 + np.random.normal(0, 1.5, size=len(df))).astype(int), 18, 70)
    df['total_purchases'] = np.clip(df['OrderCount'].astype(int), 1, 60)
    df['avg_order_value'] = np.round(np.clip(df['CashbackAmount'] * 3200 + df['OrderAmountHikeFromlastYear'] * 15000, 150000, 4500000), -3)
    df['days_since_last_purchase'] = np.clip(df['DaySinceLastOrder'].astype(int), 1, 180)
    
    cashback_q = df['CashbackAmount'].quantile([0.35, 0.70, 0.90])
    def map_membership(cb):
        if cb <= cashback_q[0.35]: return 'Đồng'
        elif cb <= cashback_q[0.70]: return 'Bạc'
        elif cb <= cashback_q[0.90]: return 'Vàng'
        else: return 'Kim Cương'
        
    df['membership_level'] = df['CashbackAmount'].apply(map_membership)
    df['used_voucher'] = (df['CouponUsed'] > 0).astype(int)
    df['satisfaction_score'] = np.clip(df['SatisfactionScore'].astype(int), 1, 5)
    
    cols = ['age', 'gender', 'total_purchases', 'avg_order_value', 'days_since_last_purchase', 'membership_level', 'used_voucher', 'satisfaction_score', 'will_return']
    return df[cols]

df = load_dataset()
print(f'Kích thước dữ liệu: {df.shape}')
display(df.head(5))

## 📌 3. TRÁNH DATA LEAKAGE & TIỀN XỬ LÝ DỮ LIỆU CHUẨN ML

In [ ]:
# === 3. CHIA TRAIN/TEST TRƯỚC ĐỂ CHỐNG DATA LEAKAGE ===
X = df.drop(columns=['will_return'])
y = df['will_return']

# Split Train (80%) / Test (20%) trước khi fit Scaler & Encoder
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_features = ['gender', 'membership_level']
num_features = ['age', 'total_purchases', 'avg_order_value', 'days_since_last_purchase', 'used_voucher', 'satisfaction_score']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

# Fit CHỈ TRÊN X_train
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

ohe_categories = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features)
feature_names = list(num_features) + list(ohe_categories)

print(f'Kích thước X_train sau pre-processing: {X_train_trans.shape}')
print(f'Kích thước X_test sau pre-processing: {X_test_trans.shape}')

## 📌 4. HUẤN LUYỆN & TỐI ƯU SIÊU THAM SỐ VỚI GRIDSEARCHCV

In [ ]:
# === 4. HUẤN LUYỆN 3 THUẬT TOÁN ML ===
results = {}

# 1. Logistic Regression
print('1. Training Logistic Regression with GridSearchCV...')
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    {'C': [0.1, 1.0, 10.0], 'solver': ['lbfgs', 'liblinear']},
    cv=5, scoring='f1', n_jobs=-1
)
grid_lr.fit(X_train_trans, y_train)
model_lr = grid_lr.best_estimator_
y_pred_lr = model_lr.predict(X_test_trans)
y_prob_lr = model_lr.predict_proba(X_test_trans)[:, 1]

results['Logistic Regression'] = {
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'precision': precision_score(y_test, y_pred_lr),
    'recall': recall_score(y_test, y_pred_lr),
    'f1': f1_score(y_test, y_pred_lr),
    'roc_auc': roc_auc_score(y_test, y_prob_lr)
}

# 2. Decision Tree
print('2. Training Decision Tree with GridSearchCV...')
grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    {'max_depth': [5, 8, 12, None], 'criterion': ['gini', 'entropy']},
    cv=5, scoring='f1', n_jobs=-1
)
grid_dt.fit(X_train_trans, y_train)
model_dt = grid_dt.best_estimator_
y_pred_dt = model_dt.predict(X_test_trans)
y_prob_dt = model_dt.predict_proba(X_test_trans)[:, 1]

results['Decision Tree'] = {
    'accuracy': accuracy_score(y_test, y_pred_dt),
    'precision': precision_score(y_test, y_pred_dt),
    'recall': recall_score(y_test, y_pred_dt),
    'f1': f1_score(y_test, y_pred_dt),
    'roc_auc': roc_auc_score(y_test, y_prob_dt)
}

# 3. Random Forest (Mô hình chính)
print('3. Training Random Forest (Ensemble) with GridSearchCV...')
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    {'n_estimators': [50, 100, 200], 'max_depth': [8, 12, None], 'max_features': ['sqrt', 'log2']},
    cv=5, scoring='f1', n_jobs=-1
)
grid_rf.fit(X_train_trans, y_train)
model_rf = grid_rf.best_estimator_
y_pred_rf = model_rf.predict(X_test_trans)
y_prob_rf = model_rf.predict_proba(X_test_trans)[:, 1]

results['Random Forest'] = {
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'precision': precision_score(y_test, y_pred_rf),
    'recall': recall_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf),
    'roc_auc': roc_auc_score(y_test, y_prob_rf)
}

print('\n=== BẢNG SO SÁNH CHỈ SỐ ĐÁNH GIÁ THỰC TẾ ===')
res_df = pd.DataFrame(results).T
res_df['accuracy'] = res_df['accuracy'] * 100
display(res_df.style.format({'accuracy': '{:.2f}%', 'precision': '{:.4f}', 'recall': '{:.4f}', 'f1': '{:.4f}', 'roc_auc': '{:.4f}'}))

## 📌 5. FEATURE IMPORTANCE (ĐỘ QUAN TRỌNG ĐẶC TRƯNG)

In [ ]:
# === 5. TRỰC QUAN HÓA FEATURE IMPORTANCE ===
importances = model_rf.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
feat_imp.tail(8).plot(kind='barh', color='#3B82F6')
plt.title('Top 8 Đặc Trưng Quan Trọng Nhất (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Score Importance')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## 📌 6. HÀM DỰ ĐOÁN VỚI CUSTOM DECISION THRESHOLD

In [ ]:
# === 6. DEMO DỰ ĐOÁN CHO KHÁCH HÀNG CỤ THỂ ===
def predict_customer(cust_dict, threshold=0.50):
    df_cust = pd.DataFrame([cust_dict])
    X_cust = preprocessor.transform(df_cust)
    prob = model_rf.predict_proba(X_cust)[0, 1]
    label = 1 if prob >= threshold else 0
    
    print(f'=== KẾT QUẢ DỰ ĐOÁN ===')
    print(f'Xác suất quay lại: {prob*100:.1f}%')
    print(f'Dự đoán nhãn (Threshold={threshold}): {"Quay lại (1)" if label==1 else "Nguy cơ rời bỏ Churn (0)"}')
    return prob, label

# Test demo
sample_customer = {
    'age': 32,
    'gender': 'Nữ',
    'total_purchases': 8,
    'avg_order_value': 850000,
    'days_since_last_purchase': 12,
    'membership_level': 'Bạc',
    'used_voucher': 1,
    'satisfaction_score': 4
}

predict_customer(sample_customer, threshold=0.50)